In [1]:
%pip install -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Definimos el modelo (DistilGPT-2 es ligero y excelente para tablas)
model_name = "distilbert/distilgpt2"
local_model_path = "./pretrained-distilgpt2"

synthetizer_path="distilgpt2-distil.pickle"
synth_data_output_path="distilgpt2_synth_out.csv"

print(f"Descargando {model_name}...")

# Descargamos Tokenizer y Modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Guardamos en local
tokenizer.save_pretrained(local_model_path)
model.save_pretrained(local_model_path)

print(f"¡Listo! Copia la carpeta '{local_model_path}' a tu máquina sin internet.")

/home/aperez/workspace/synthetic-data-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando distilbert/distilgpt2...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]

¡Listo! Copia la carpeta './pretrained-distilgpt2' a tu máquina sin internet.


In [3]:
import pandas as pd
import numpy as np

from be_great import GReaT
from sdv.metadata import Metadata
from sdv.evaluation import evaluate_quality

# 1. Cargar el dataset adjunto
df_real = pd.read_csv('../../datasources/bank_full_dataset/bank-full.csv', sep=';')

# Para la PoC, tomamos una muestra representativa para optimizar tiempos de GPU/CPU
# En producción (Vertex AI), usaríamos el dataset completo.
df_train = df_real.sample(n=1000, random_state=42)
metadata = Metadata.detect_from_dataframes(
    data={'bank_full': df_real}
)

print(f"Dataset cargado: {df_real.shape[0]} registros.")
print("Columnas detectadas:", df_real.columns.tolist())

num_cols = df_real.select_dtypes(include=[np.number]).columns
print("Columnas numericas:", num_cols)
cat_cols = df_real.select_dtypes(exclude=[np.number]).columns
print("Columnas categoricas:", cat_cols)

Dataset cargado: 45211 registros.
Columnas detectadas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Columnas numericas: Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')
Columnas categoricas: Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')


In [4]:
import pandas as pd
import torch
from be_great import GReaT

# 1. Ruta donde copiaste la carpeta

# 3. Inicializamos GReaT apuntando a la ruta local
# Importante: llm=local_model_path evita que intente conectar a internet
model = GReaT(
    llm=local_model_path,  
    float_precision=2,  # Limit floating-point precision to 3 decimal places
    batch_size=16,     # Use smaller batch size for small datasets
    epochs=16,          # Train for more epochs with small data
    bf16=torch.cuda.is_available()  # bf16 solo con GPU: transformers falla si se activa en CPU
)

# Entrenamiento
model.fit(df_train)
model.save(synthetizer_path)


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2813.95it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'loss': '1.345', 'grad_norm': '3.641', 'learning_rate': '2.525e-05', 'epoch': '7.937'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s]


{'loss': '1.04', 'grad_norm': '4.01', 'learning_rate': '4.464e-07', 'epoch': '15.87'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


{'train_runtime': '91.89', 'train_samples_per_second': '174.1', 'train_steps_per_second': '10.97', 'train_loss': '1.191', 'epoch': '16'}


/home/aperez/workspace/synthetic-data-learning/.venv/lib/python3.12/site-packages/be_great/great.py:1209: UserWarning: Directory distilgpt2-distil.pickle already exists and is overwritten now.
  warnings.warn(f"Directory {path} already exists and is overwritten now.")


In [5]:
# 3. Generación de datos sintéticos
# Vamos a generar 1000 registros nuevos "soñados" por el modelo
df_synthetic = model.sample(n_samples=1000, guided_sampling=True)

# Limpieza rápida: GREAT a veces genera strings con espacios extra
df_synthetic = df_synthetic.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

print("Muestra de datos sintéticos generados:")
display(df_synthetic.head())

df_synthetic.to_csv(synth_data_output_path)

[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


Muestra de datos sintéticos generados:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,53.0,blue-collar,married,secondary,no,0.0,no,yes,cellular,14.0,nov,139,2.0,-1,0.0,unknown,no
1,32.0,blue-collar,single,secondary,no,1164.0,yes,no,cellular,19.0,jul,24,2.0,-1,0.0,unknown,no
2,51.0,management,married,secondary,no,1478.0,no,no,cellular,19.0,-1,218,1.0,-1,0.0,unknown,no
3,48.0,management,married,secondary,no,24.0,no,no,telephone,27.0,mayan,1305,1.0,-1,0.0,unknown,yes
4,43.0,blue-collar,married,secondary,no,772.0,yes,no,cellular,28.0,mayan,925,2.0,-1,0.0,unknown,no


In [6]:
# 5. Función de reporte de calidad
def reporte_calidad(real, synth):
    print("\n" + "="*40)
    print("   REPORTE DE FIDELIDAD ESTADÍSTICA")
    print("="*40 + "\n")
      
    # Test de Proporciones para Categóricas (Diferencia Media)
    cat_cols = real.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        f_real = real[col].value_counts(normalize=True).sort_index()
        f_synth = synth[col].value_counts(normalize=True).sort_index()
        diff = (f_real - f_synth).abs().mean()
        status = "✅ FIEL" if diff < 0.05 else "❌ DESVIADO"
        print(f"Prop. Diff [{col:10}]: {diff:.4f} {status}")

reporte_calidad(df_real, df_synthetic)


   REPORTE DE FIDELIDAD ESTADÍSTICA

Prop. Diff [job       ]: 0.0828 ❌ DESVIADO
Prop. Diff [marital   ]: 0.0947 ❌ DESVIADO
Prop. Diff [education ]: 0.0336 ✅ FIEL
Prop. Diff [default   ]: 0.0145 ✅ FIEL
Prop. Diff [housing   ]: 0.1632 ❌ DESVIADO
Prop. Diff [loan      ]: 0.0692 ❌ DESVIADO
Prop. Diff [contact   ]: 0.1708 ❌ DESVIADO
Prop. Diff [month     ]: 0.0291 ✅ FIEL
Prop. Diff [poutcome  ]: 0.0860 ❌ DESVIADO
Prop. Diff [y         ]: 0.0760 ❌ DESVIADO


In [7]:
import torch
import gc

# 1. Eliminar el objeto del modelo (y cualquier otro pesado)
if 'model' in locals():
    del model
if 'df_synthetic' in locals():
    del df_synthetic

# 2. Forzar la recolección de basura de Python
gc.collect()

# 3. Vaciar el caché de PyTorch (esto libera la VRAM físicamente)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
print("Memoria de GPU liberada.")

Memoria de GPU liberada.
